# 🐼 Pandas Complete Guide
A hands-on notebook covering pandas from basics to intermediate level.

**Contents**
1. Introduction & Theory
2. Creating Series (list, dict, scalar, NumPy)
3. Creating DataFrames (list, dict, list of dicts, ...)
4. Importing Files (CSV, Excel, JSON, SQL, HTML, Parquet, text, URL...)
5. Inspecting Data
6. Selecting & Filtering (`[]`, `loc`, `iloc`, `query`)
7. Adding / Modifying / Dropping Columns
8. Handling Null Values (detect, drop, fill)
9. Imputation: Mean vs Median vs Mode (and when to use)
10. Duplicates
11. Sorting
12. GroupBy, Aggregation, Pivot
13. Joins: `merge`, `join`, `concat`
14. String & DateTime functions
15. Other useful functions
16. Exporting Data
17. Practice Questions
18. Solutions

## 1. Introduction & Theory

**Pandas** is Python's main library for data analysis and manipulation. It is built on top of NumPy.

It has two core data structures:

| Structure | Dimensions | Description |
|---|---|---|
| **Series** | 1-D | A labelled column of data (values + index) |
| **DataFrame** | 2-D | A table of rows and columns; each column is a Series |

**Key ideas**
- **Index**: labels for rows (default `0, 1, 2, ...`).
- **dtype**: each column has one data type (`int64`, `float64`, `object`/`string`, `bool`, `datetime64`, `category`).
- **Missing values** are represented by `NaN` (float), `NaT` (datetime) or `pd.NA`.
- Most methods **return a new object** instead of changing the original, unless you assign the result back (avoid `inplace=True` where possible).
- Operations are **vectorised**: avoid Python loops, use column-wise operations.

In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

## 2. Creating a Series

A **Series** = 1-D array + an index. You can create it from a list, dictionary, scalar or NumPy array.

### 2.1 From a list

In [ ]:
s = pd.Series([10, 20, 30, 40])
print(s)

Custom index and name:

In [ ]:
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='scores')
print(s)
print("\nValue at 'b':", s['b'])

### 2.2 From a dictionary
Dictionary **keys become the index**, values become the data.

In [ ]:
d = {'Mon': 5, 'Tue': 8, 'Wed': 3}
s = pd.Series(d)
print(s)

### 2.3 From a scalar and from NumPy

In [ ]:
print(pd.Series(7, index=['x', 'y', 'z']))      # scalar is broadcast
print(pd.Series(np.arange(5)))                  # from NumPy array
print(pd.Series(np.random.randn(3)))            # random values

### 2.4 Series attributes & operations

In [ ]:
s = pd.Series([3, 7, 1, 9, 5], index=list('abcde'))
print("values :", s.values)
print("index  :", s.index)
print("dtype  :", s.dtype)
print("size   :", s.size)
print("sum/mean/max:", s.sum(), s.mean(), s.max())
print("\nVectorised ops:\n", s * 2)
print("\nFilter > 4:\n", s[s > 4])
print("\nSlicing:\n", s['b':'d'])        # label slice includes the end

## 3. Creating a DataFrame

A **DataFrame** is a 2-D labelled table. Common ways to build one:

### 3.1 From a dictionary of lists  (most common)
Keys → column names, lists → column values.

In [ ]:
data = {
    'name':   ['Asha', 'Ben', 'Chitra', 'Dev'],
    'age':    [25, 30, 28, 35],
    'city':   ['Delhi', 'Mumbai', 'Pune', 'Delhi'],
    'salary': [50000, 65000, 58000, 72000]
}
df = pd.DataFrame(data)
df

### 3.2 From a list of lists  (give column names)

In [ ]:
rows = [
    ['Asha', 25, 'Delhi'],
    ['Ben', 30, 'Mumbai'],
    ['Chitra', 28, 'Pune']
]
pd.DataFrame(rows, columns=['name', 'age', 'city'])

### 3.3 From a list of dictionaries  (each dict = one row; missing keys become NaN)

In [ ]:
records = [
    {'name': 'Asha', 'age': 25},
    {'name': 'Ben', 'age': 30, 'city': 'Mumbai'},
    {'name': 'Chitra', 'city': 'Pune'}
]
pd.DataFrame(records)

### 3.4 From a dictionary of Series  (aligned on index)

In [ ]:
s1 = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20], index=['a', 'c'])
pd.DataFrame({'col1': s1, 'col2': s2})

### 3.5 From a NumPy array

In [ ]:
arr = np.random.randint(0, 100, size=(3, 4))
pd.DataFrame(arr, columns=list('ABCD'), index=['r1', 'r2', 'r3'])

### 3.6 From a list of tuples / `zip`

In [ ]:
names = ['A', 'B', 'C']
marks = [90, 80, 70]
pd.DataFrame(list(zip(names, marks)), columns=['student', 'marks'])

### 3.7 Series ↔ DataFrame

In [ ]:
print(type(df['age']))              # a column is a Series
print(type(df[['age']]))            # double brackets keep a DataFrame
df['age'].to_frame()

## 4. Importing Files (Different Ways)

| Function | File type |
|---|---|
| `pd.read_csv()` | CSV / delimited text |
| `pd.read_table()` | tab-separated text |
| `pd.read_excel()` | Excel (`.xlsx`, `.xls`) – needs `openpyxl` |
| `pd.read_json()` | JSON |
| `pd.read_sql()` / `read_sql_query()` | SQL databases |
| `pd.read_html()` | Tables on web pages – needs `lxml` |
| `pd.read_parquet()` | Parquet – needs `pyarrow` |
| `pd.read_pickle()` | Python pickle |
| `pd.read_xml()` | XML |
| `pd.read_clipboard()` | Copied text |
| `pd.read_fwf()` | Fixed-width files |

First, let's create sample files so every example in this notebook can run.

In [ ]:
import os
os.makedirs('sample_data', exist_ok=True)

sample = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'name': ['Asha', 'Ben', 'Chitra', 'Dev', 'Esha'],
    'dept': ['HR', 'IT', 'IT', 'Sales', 'HR'],
    'salary': [50000, 65000, 58000, 72000, 49000],
    'joined': ['2020-01-15', '2019-03-10', '2021-07-01', '2018-11-20', '2022-02-05']
})
sample.to_csv('sample_data/emp.csv', index=False)
sample.to_csv('sample_data/emp_semicolon.csv', index=False, sep=';')
sample.to_csv('sample_data/emp.txt', index=False, sep='\t')
sample.to_json('sample_data/emp.json', orient='records')
sample.to_pickle('sample_data/emp.pkl')
print(os.listdir('sample_data'))

### 4.1 CSV – basic and with useful parameters

In [ ]:
df = pd.read_csv('sample_data/emp.csv')
df

In [ ]:
# Frequently used read_csv parameters
df = pd.read_csv(
    'sample_data/emp.csv',
    usecols=['id', 'name', 'salary', 'joined'],  # load only some columns
    index_col='id',                              # use a column as index
    parse_dates=['joined'],                      # convert to datetime
    dtype={'salary': 'float64'},                 # force a dtype
    nrows=4,                                     # read only first 4 rows
    na_values=['NA', 'missing', '-']             # extra strings to treat as NaN
)
print(df.dtypes)
df

In [ ]:
# skiprows, header, names, custom separator
pd.read_csv('sample_data/emp_semicolon.csv', sep=';').head(2)

In [ ]:
# Read huge files in chunks
total = 0
for chunk in pd.read_csv('sample_data/emp.csv', chunksize=2):
    total += chunk['salary'].sum()
print("Total salary:", total)

### 4.2 Tab/Text files

In [ ]:
pd.read_table('sample_data/emp.txt').head(2)         # default sep is tab

### 4.3 JSON

In [ ]:
pd.read_json('sample_data/emp.json').head(2)

### 4.4 From a string (`StringIO`) – handy for tests

In [ ]:
from io import StringIO
text = '''a,b,c
1,2,3
4,5,6'''
pd.read_csv(StringIO(text))

### 4.5 Excel

In [ ]:
try:
    sample.to_excel('sample_data/emp.xlsx', index=False, sheet_name='Employees')
    print(pd.read_excel('sample_data/emp.xlsx', sheet_name='Employees').head(2))
    # all sheets: pd.read_excel(path, sheet_name=None) -> dict of DataFrames
except ImportError as e:
    print("Install openpyxl:  pip install openpyxl\n", e)

### 4.6 SQL (SQLite example)

In [ ]:
import sqlite3
conn = sqlite3.connect('sample_data/emp.db')
sample.to_sql('employees', conn, if_exists='replace', index=False)
pd.read_sql("SELECT name, salary FROM employees WHERE salary > 55000", conn)

In [ ]:
conn.close()

### 4.7 Parquet, Pickle, HTML, URL, Clipboard

In [ ]:
# Pickle
print(pd.read_pickle('sample_data/emp.pkl').shape)

# Parquet (pip install pyarrow)
try:
    sample.to_parquet('sample_data/emp.parquet')
    print(pd.read_parquet('sample_data/emp.parquet').shape)
except ImportError as e:
    print("Install pyarrow: pip install pyarrow")

# Read directly from a URL (needs internet) – examples, not run:
# pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv')
# pd.read_html('https://en.wikipedia.org/wiki/List_of_countries_by_population')[0]
# pd.read_clipboard()   # after copying a table

### 4.8 Reading from Google Colab / Upload (reference)
```python
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('your_file.csv')
```

## 5. Inspecting Data
We'll use a richer dataset that includes **missing values** and duplicates.

In [ ]:
emp = pd.DataFrame({
    'emp_id':  [101, 102, 103, 104, 105, 106, 107, 108],
    'name':    ['Asha', 'Ben', 'Chitra', 'Dev', 'Esha', 'Farhan', 'Gita', 'Hari'],
    'dept':    ['HR', 'IT', 'IT', 'Sales', None, 'IT', 'HR', 'Sales'],
    'age':     [25, 30, np.nan, 35, 28, 41, np.nan, 33],
    'salary':  [50000, 65000, 58000, np.nan, 49000, 120000, 52000, 61000],
    'rating':  [4.5, 3.8, 4.1, 3.9, np.nan, 4.7, 4.0, np.nan],
    'city':    ['Delhi', 'Mumbai', 'Pune', 'Delhi', 'Pune', 'Mumbai', 'Delhi', 'Pune']
})
emp

In [ ]:
print("Shape  :", emp.shape)         # (rows, columns)
print("Size   :", emp.size)
print("Columns:", emp.columns.tolist())
print("Index  :", emp.index)
print("\nDtypes:\n", emp.dtypes)

In [ ]:
emp.head(3)

In [ ]:
emp.tail(3)

In [ ]:
emp.sample(3, random_state=1)

In [ ]:
emp.info()

In [ ]:
emp.describe()                  # numeric columns

In [ ]:
emp.describe(include='object')  # text columns

In [ ]:
print(emp['dept'].value_counts())              # counts of each value
print(emp['dept'].value_counts(dropna=False))  # include NaN
print("Unique cities:", emp['city'].unique())
print("No. of unique:", emp['city'].nunique())

In [ ]:
# Common statistics
print(emp['salary'].sum(), emp['salary'].mean(), emp['salary'].median())
print(emp['salary'].min(), emp['salary'].max(), emp['salary'].std())
print(emp[['age', 'salary']].corr())
print(emp['salary'].quantile([0.25, 0.5, 0.75]))

## 6. Selecting & Filtering

### 6.1 Column selection

In [ ]:
emp['name']                 # Series
emp[['name', 'salary']]     # DataFrame

### 6.2 `loc` (label-based) and `iloc` (position-based)

In [ ]:
print(emp.loc[0])                              # row with label 0
print(emp.loc[0:2, ['name', 'salary']])        # labels: end INCLUDED
print(emp.iloc[0:2, 0:3])                      # positions: end EXCLUDED
print(emp.iloc[-1])                            # last row
print(emp.at[2, 'name'], emp.iat[2, 1])        # fast scalar access

### 6.3 Boolean filtering

In [ ]:
emp[emp['salary'] > 55000]

In [ ]:
# Multiple conditions: use & (and), | (or), ~ (not) with parentheses
emp[(emp['dept'] == 'IT') & (emp['salary'] > 60000)]

In [ ]:
print(emp[emp['city'].isin(['Delhi', 'Pune'])])
print(emp[emp['salary'].between(50000, 60000)])
print(emp[~emp['dept'].isin(['IT'])])

### 6.4 `query()` – SQL-like filtering

In [ ]:
min_sal = 55000
emp.query("dept == 'IT' and salary > @min_sal")

### 6.5 Set and reset index

In [ ]:
e2 = emp.set_index('emp_id')
print(e2.loc[103, 'name'])
e2.reset_index().head(2)

## 7. Adding / Modifying / Dropping Columns

In [ ]:
df = emp.copy()    # always work on a copy when experimenting

# add column
df['bonus'] = df['salary'] * 0.10
# conditional column
df['level'] = np.where(df['salary'] > 60000, 'High', 'Low')
# assign (returns a new df, chainable)
df = df.assign(salary_k=df['salary'] / 1000)
df.head(3)

In [ ]:
# apply / map
df['name_len'] = df['name'].apply(len)
df['city_code'] = df['city'].map({'Delhi': 'DEL', 'Mumbai': 'BOM', 'Pune': 'PNQ'})
df['salary_grade'] = df['salary'].apply(lambda x: 'A' if x > 60000 else 'B')
# row-wise apply
df['summary'] = df.apply(lambda r: f"{r['name']} ({r['city']})", axis=1)
df[['name', 'name_len', 'city_code', 'salary_grade', 'summary']].head(3)

In [ ]:
# rename, drop, replace, astype
df = df.rename(columns={'emp_id': 'id'})
df = df.drop(columns=['bonus', 'salary_k', 'name_len', 'summary'])
df['city'] = df['city'].replace({'Delhi': 'New Delhi'})
df['id'] = df['id'].astype('str')
df['dept'] = df['dept'].astype('category')
print(df.dtypes)
df.head(3)

In [ ]:
# Drop rows, reorder columns, insert at position
df = df.drop(index=[0, 1])
df.insert(1, 'country', 'India')
df[['id', 'country', 'name']].head(3)

## 8. Handling Null (Missing) Values

### Theory
Missing data appears as `NaN` (Not a Number), `None`, `NaT` or `pd.NA`.

**Why does data go missing?**
- **MCAR** – Missing Completely At Random (no pattern)
- **MAR** – Missing At Random (depends on other columns)
- **MNAR** – Missing Not At Random (depends on the missing value itself, e.g., high earners hide salary)

**Strategies**
1. **Drop** rows/columns (`dropna`) – fine when few values are missing.
2. **Fill / Impute** (`fillna`) – constant, mean, median, mode, forward/backward fill, interpolation.
3. **Flag** – add an indicator column (`is_missing`) so the model/analysis knows a value was imputed.

### 8.1 Detecting nulls

In [ ]:
print(emp.isnull().sum())                          # count per column
print("\nPercentage missing:\n", (emp.isnull().mean() * 100).round(1))
print("\nTotal nulls:", emp.isnull().sum().sum())
emp[emp.isnull().any(axis=1)]                      # rows with any null

In [ ]:
# notna / notnull are the opposites
emp['salary'].notna().sum()

### 8.2 Dropping nulls – `dropna`

In [ ]:
print(emp.dropna().shape)                          # drop rows with ANY null
print(emp.dropna(how='all').shape)                 # drop rows where ALL are null
print(emp.dropna(subset=['salary']).shape)         # only check specific columns
print(emp.dropna(thresh=6).shape)                  # keep rows with >= 6 non-null values
print(emp.dropna(axis=1).columns.tolist())         # drop COLUMNS having any null

### 8.3 Filling nulls – `fillna`

In [ ]:
df = emp.copy()
df['dept'] = df['dept'].fillna('Unknown')          # constant
df['rating'] = df['rating'].fillna(0)
df.isnull().sum()

In [ ]:
# Different value per column using a dictionary
emp.fillna({'dept': 'Unknown', 'age': emp['age'].median(), 'salary': emp['salary'].mean()})

### 8.4 Forward fill, backward fill, interpolation (great for time series)

In [ ]:
ts = pd.Series([1, np.nan, np.nan, 4, np.nan, 6])
print(pd.DataFrame({
    'original': ts,
    'ffill': ts.ffill(),                # copy last valid value forward
    'bfill': ts.bfill(),                # copy next valid value backward
    'interpolate': ts.interpolate()     # linear interpolation
}))

## 9. Imputation: Mean vs Median vs Mode

| Method | Use for | Use when | Avoid when |
|---|---|---|---|
| **Mean** | Numeric | Data is roughly **symmetric/normal**, **no outliers** (e.g., heights, exam marks) | Skewed data or outliers (mean gets pulled) |
| **Median** | Numeric | Data is **skewed** or has **outliers** (e.g., salary, house prices, income) | Rarely a problem; slightly distorts variance |
| **Mode** | Categorical (also discrete numeric) | Column is **categorical** (city, dept, gender) or has a clear most-frequent value | Many categories with similar frequencies |

**Other options**
- Constant like `'Unknown'` / `-1` – when "missing" itself is meaningful.
- Group-wise mean/median – when other columns explain the value (e.g., median salary **per department**).
- ffill / bfill / interpolate – for time series.
- KNN / model-based imputation (`sklearn`) – advanced.

> **Rule of thumb:** Mean → normal numeric. Median → skewed/outlier numeric. Mode → categorical.

> **Tip:** if more than ~40–50% of a column is missing, consider dropping the column instead.

Check the distribution first (skewness & outliers help decide mean vs median):

In [ ]:
print("Skewness of salary:", round(emp['salary'].skew(), 2))   # far from 0 => skewed
print("Mean   :", emp['salary'].mean())
print("Median :", emp['salary'].median())
emp['salary'].plot(kind='box', title='Salary – note the outlier (120000)');

Here the mean (~65,000) is pulled up by the 120,000 outlier, so **median** is the better choice for `salary`.

### 9.1 Mean imputation

In [ ]:
df = emp.copy()
df['age'] = df['age'].fillna(df['age'].mean())
df[['name', 'age']]

### 9.2 Median imputation

In [ ]:
df = emp.copy()
df['salary'] = df['salary'].fillna(df['salary'].median())
df[['name', 'salary']]

### 9.3 Mode imputation  (`mode()` returns a Series → take `[0]`)

In [ ]:
print(emp['dept'].mode())
df = emp.copy()
df['dept'] = df['dept'].fillna(df['dept'].mode()[0])
df[['name', 'dept']]

### 9.4 Group-wise imputation (median salary per department)

In [ ]:
df = emp.copy()
df['salary'] = df['salary'].fillna(df.groupby('dept')['salary'].transform('median'))
df[['name', 'dept', 'salary']]

### 9.5 Complete imputation pipeline + missing-flag column

In [ ]:
df = emp.copy()

# flag before imputing
df['salary_was_missing'] = df['salary'].isnull()

num_cols = df.select_dtypes(include='number').columns.drop('emp_id')
cat_cols = df.select_dtypes(include='object').columns

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())          # numeric -> median
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])         # categorical -> mode

print(df.isnull().sum().sum(), "nulls left")
df

### 9.6 Same thing with scikit-learn's `SimpleImputer` (optional)

In [ ]:
try:
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='median')          # 'mean', 'median', 'most_frequent', 'constant'
    out = emp.copy()
    out[['age', 'salary', 'rating']] = imp.fit_transform(out[['age', 'salary', 'rating']])
    print(out[['age', 'salary', 'rating']])
except ImportError:
    print("pip install scikit-learn to try this")

## 10. Duplicates

In [ ]:
dup = pd.DataFrame({'a': [1, 1, 2, 2, 3], 'b': ['x', 'x', 'y', 'z', 'w']})
print(dup.duplicated())                          # True for repeated rows (after first)
print(dup.duplicated(subset=['a']).sum())        # based on a column
print(dup.drop_duplicates())                     # keep='first' (default)
print(dup.drop_duplicates(subset=['a'], keep='last'))

## 11. Sorting

In [ ]:
emp.sort_values('salary')                                  # ascending (NaN last)

In [ ]:
emp.sort_values('salary', ascending=False)                 # descending

In [ ]:
# multiple columns, different order each
emp.sort_values(['dept', 'salary'], ascending=[True, False], na_position='first')

In [ ]:
# sort by index
emp.sort_index(ascending=False).head(3)

In [ ]:
# top-N / bottom-N, rank
print(emp.nlargest(3, 'salary')[['name', 'salary']])
print(emp.nsmallest(2, 'age')[['name', 'age']])
emp['salary_rank'] = emp['salary'].rank(ascending=False, method='dense')
emp[['name', 'salary', 'salary_rank']].sort_values('salary_rank')

In [ ]:
# custom sort order using an ordered categorical
order = pd.CategoricalDtype(['Sales', 'IT', 'HR'], ordered=True)
tmp = emp.dropna(subset=['dept']).copy()
tmp['dept'] = tmp['dept'].astype(order)
tmp.sort_values('dept')[['name', 'dept']]

In [ ]:
emp = emp.drop(columns='salary_rank')   # clean up

## 12. GroupBy, Aggregation & Pivot
`groupby` follows **split → apply → combine**.

In [ ]:
print(emp.groupby('dept')['salary'].mean())
print(emp.groupby('city')['salary'].agg(['count', 'min', 'max', 'mean']))

In [ ]:
# named aggregation
emp.groupby('dept').agg(
    headcount=('emp_id', 'count'),
    avg_salary=('salary', 'mean'),
    max_age=('age', 'max')
).reset_index()

In [ ]:
# multiple group keys
emp.groupby(['dept', 'city'])['salary'].sum().reset_index()

In [ ]:
# transform keeps original shape; filter keeps groups meeting a condition
emp['dept_avg'] = emp.groupby('dept')['salary'].transform('mean')
print(emp[['name', 'dept', 'salary', 'dept_avg']].head(4))
print(emp.groupby('dept').filter(lambda g: len(g) >= 3)[['name', 'dept']])
emp = emp.drop(columns='dept_avg')

In [ ]:
# pivot_table & crosstab
print(pd.pivot_table(emp, values='salary', index='dept', columns='city', aggfunc='mean', fill_value=0))
print(pd.crosstab(emp['dept'], emp['city']))

## 13. Joins: `merge`, `join`, `concat`

### Theory – types of joins
| Join | Keeps |
|---|---|
| **inner** | only keys present in **both** tables |
| **left** | all rows from left + matching from right |
| **right** | all rows from right + matching from left |
| **outer** | all rows from **both** (NaN where no match) |
| **cross** | every combination (Cartesian product) |

In [ ]:
employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'name': ['Asha', 'Ben', 'Chitra', 'Dev'],
    'dept_id': [10, 20, 10, 40]
})
departments = pd.DataFrame({
    'dept_id': [10, 20, 30],
    'dept_name': ['HR', 'IT', 'Finance']
})
print(employees); print(departments)

In [ ]:
print("INNER\n", pd.merge(employees, departments, on='dept_id', how='inner'))
print("\nLEFT\n", pd.merge(employees, departments, on='dept_id', how='left'))
print("\nRIGHT\n", pd.merge(employees, departments, on='dept_id', how='right'))
print("\nOUTER\n", pd.merge(employees, departments, on='dept_id', how='outer'))

In [ ]:
# indicator shows where each row came from
pd.merge(employees, departments, on='dept_id', how='outer', indicator=True)

In [ ]:
# Different key names, and suffixes for overlapping columns
a = pd.DataFrame({'eid': [1, 2], 'val': [10, 20]})
b = pd.DataFrame({'emp': [1, 2], 'val': [100, 200]})
pd.merge(a, b, left_on='eid', right_on='emp', suffixes=('_a', '_b'))

In [ ]:
# join() merges on the INDEX
l = employees.set_index('emp_id')
r = pd.DataFrame({'salary': [50000, 60000, 70000]}, index=[1, 2, 3])
l.join(r, how='left')

### concat – stack DataFrames

In [ ]:
d1 = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
d2 = pd.DataFrame({'a': [5, 6], 'b': [7, 8]})
print(pd.concat([d1, d2], ignore_index=True))      # vertical (rows)
print(pd.concat([d1, d2], axis=1))                 # horizontal (columns)

In [ ]:
# validate catches unexpected duplicate keys
try:
    pd.merge(employees, departments, on='dept_id', validate='one_to_one')
    print("keys are unique on both sides")
except Exception as e:
    print("Validation error:", e)

## 14. String & DateTime Functions

### 14.1 String methods (`.str`)

In [ ]:
s = pd.Series(['  Alice Smith ', 'bob JONES', None, 'Carol  White'])
print(s.str.strip().str.lower())
print(s.str.upper())
print(s.str.len())
print(s.str.contains('smith', case=False))
print(s.str.split().str[0])
print(s.str.replace(' ', '_', regex=False))

### 14.2 DateTime (`.dt`)

In [ ]:
dates = pd.DataFrame({'d': ['2024-01-15', '2024-03-20', '2024-12-25']})
dates['d'] = pd.to_datetime(dates['d'])
dates['year'] = dates['d'].dt.year
dates['month'] = dates['d'].dt.month
dates['day_name'] = dates['d'].dt.day_name()
dates['quarter'] = dates['d'].dt.quarter
dates['days_since'] = (pd.Timestamp('2025-01-01') - dates['d']).dt.days
dates

In [ ]:
# date ranges and resampling
rng = pd.date_range('2024-01-01', periods=6, freq='D')
ts = pd.Series(range(6), index=rng)
print(ts.resample('2D').sum())

## 15. Other Useful Functions

In [ ]:
# cut / qcut – binning
ages = pd.Series([5, 17, 25, 40, 65, 80])
print(pd.cut(ages, bins=[0, 18, 60, 100], labels=['child', 'adult', 'senior']))
print(pd.qcut(ages, q=3, labels=['low', 'mid', 'high']))

In [ ]:
# One-hot encoding
pd.get_dummies(emp[['dept', 'city']].dropna(), prefix=['d', 'c'], dtype=int).head(4)

In [ ]:
# cumulative, shift, diff, rolling, pct_change
x = pd.Series([10, 20, 15, 30, 25])
print(pd.DataFrame({
    'x': x, 'cumsum': x.cumsum(), 'shift': x.shift(1),
    'diff': x.diff(), 'pct': x.pct_change().round(2),
    'roll_mean_3': x.rolling(3).mean()
}))

In [ ]:
# clip, where / mask, isin, idxmax / idxmin
print(emp['salary'].clip(lower=50000, upper=70000).tolist())
print(emp.loc[emp['salary'].idxmax(), 'name'], "earns the most")
emp['salary'].where(emp['salary'] > 55000, other=0).tolist()

In [ ]:
# melt (wide -> long) and pivot (long -> wide)
wide = pd.DataFrame({'name': ['A', 'B'], 'math': [90, 80], 'sci': [85, 95]})
long = wide.melt(id_vars='name', var_name='subject', value_name='score')
print(long)
print(long.pivot(index='name', columns='subject', values='score'))

In [ ]:
# Memory optimisation, copy vs view, method chaining
result = (
    emp
    .dropna(subset=['salary'])
    .assign(salary_k=lambda d: d['salary'] / 1000)
    .query("salary_k > 50")
    .sort_values('salary_k', ascending=False)
    [['name', 'dept', 'salary_k']]
)
result

In [ ]:
# Display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 2)
pd.reset_option('display.precision')

## 16. Exporting Data

In [ ]:
emp.to_csv('sample_data/out.csv', index=False)
emp.to_json('sample_data/out.json', orient='records', indent=2)
emp.to_pickle('sample_data/out.pkl')
try:
    emp.to_excel('sample_data/out.xlsx', index=False)
except ImportError:
    print("pip install openpyxl")
print(emp.head(2).to_dict(orient='records'))     # to dict
print(emp.head(2).to_markdown() if hasattr(emp, 'to_markdown') and __import__('importlib').util.find_spec('tabulate') else "pip install tabulate for to_markdown")

## 17. Practice Questions

Use the DataFrame below for the questions. Try each one before looking at the solutions in Section 18.

In [ ]:
sales = pd.DataFrame({
    'order_id': range(1, 11),
    'customer': ['Amit', 'Bela', 'Chan', 'Amit', 'Dina', 'Bela', 'Esha', 'Chan', None, 'Amit'],
    'product': ['Laptop', 'Phone', 'Phone', 'Tablet', 'Laptop', 'Tablet', 'Phone', 'Laptop', 'Tablet', 'Phone'],
    'qty': [1, 2, 1, 3, np.nan, 2, 1, 1, 2, np.nan],
    'price': [800, 500, 500, 300, 800, 300, 500, 800, 300, 500],
    'date': pd.to_datetime(['2024-01-05', '2024-01-12', '2024-02-03', '2024-02-15', '2024-03-01',
                            '2024-03-10', '2024-03-22', '2024-04-02', '2024-04-18', '2024-04-25'])
})
customers = pd.DataFrame({
    'customer': ['Amit', 'Bela', 'Chan', 'Dina', 'Farid'],
    'city': ['Delhi', 'Pune', 'Mumbai', 'Delhi', 'Chennai']
})
sales

### Basic
1. Display the shape, column names and data types of `sales`.
2. Show the first 3 and last 2 rows.
3. How many unique products are there? Show the count of orders for each product.
4. Select the `customer` and `price` columns for orders where `price` > 400.

### Null values
5. Count the null values in every column. What percentage of `qty` is missing?
6. Fill missing `customer` with `'Unknown'`.
7. Fill missing `qty` using the **median** of `qty`. Explain why median or mean may be preferred here.
8. Fill missing `qty` using the **mode** instead. When is mode a better choice than mean?
9. Fill missing `qty` with the **median qty of the same product** (group-wise).
10. Drop all rows where `qty` is null. How many rows remain?

### Transformations
11. Create a `revenue` column = `qty * price` (after imputing `qty`).
12. Create a column `size` with `'Big'` if revenue >= 1000 else `'Small'`.
13. Extract month name from `date` into a new column.
14. Rename `qty` to `quantity` and drop `order_id`.

### Sorting
15. Sort by `revenue` descending and show the top 3 orders.
16. Sort by `product` ascending and `revenue` descending.

### GroupBy
17. Find total revenue per product.
18. Find average revenue per customer, sorted from high to low.
19. Create a pivot table: rows = product, columns = month, values = total revenue.

### Joins
20. Merge `sales` with `customers` using a **left join** on `customer`. Which orders have no city?
21. Do an **inner join**. How many rows are in the result?
22. Do an **outer join** with `indicator=True`. Which customers have no orders?

### Challenge
23. Write one method chain that: fills `qty` with median → creates `revenue` → keeps only `Phone` and `Laptop` → groups by product → returns total revenue.
24. Find the customer who spent the most in total.
25. Read `sample_data/emp.csv` with `id` as index, `joined` parsed as dates, and only 3 columns of your choice.

## 18. Solutions
Run `sales_clean` creation first (Q6–Q11) because later solutions build on it.

In [ ]:
# Q1
print(sales.shape); print(sales.columns.tolist()); print(sales.dtypes)
# Q2
print(sales.head(3)); print(sales.tail(2))
# Q3
print(sales['product'].nunique()); print(sales['product'].value_counts())
# Q4
sales.loc[sales['price'] > 400, ['customer', 'price']]

In [ ]:
# Q5
print(sales.isnull().sum())
print("qty % missing:", sales['qty'].isnull().mean() * 100)

In [ ]:
# Q6-Q11
sales_clean = sales.copy()
sales_clean['customer'] = sales_clean['customer'].fillna('Unknown')                  # Q6
sales_clean['qty'] = sales_clean['qty'].fillna(sales_clean['qty'].median())          # Q7
# Median is robust to outliers (use when skewed); mean when data is symmetric.

# Q8 - mode: use for categorical or discrete values with a clear most-common value
qty_mode = sales['qty'].fillna(sales['qty'].mode()[0])
# Q9 - group-wise median
qty_grp = sales['qty'].fillna(sales.groupby('product')['qty'].transform('median'))
# Q10
print("Rows after dropna on qty:", len(sales.dropna(subset=['qty'])))
# Q11
sales_clean['revenue'] = sales_clean['qty'] * sales_clean['price']
sales_clean

In [ ]:
# Q12-Q14
sales_clean['size'] = np.where(sales_clean['revenue'] >= 1000, 'Big', 'Small')       # Q12
sales_clean['month'] = sales_clean['date'].dt.month_name()                           # Q13
sales_clean = sales_clean.rename(columns={'qty': 'quantity'}).drop(columns='order_id')  # Q14
sales_clean.head()

In [ ]:
# Q15
print(sales_clean.sort_values('revenue', ascending=False).head(3))
# Q16
sales_clean.sort_values(['product', 'revenue'], ascending=[True, False])

In [ ]:
# Q17
print(sales_clean.groupby('product')['revenue'].sum())
# Q18
print(sales_clean.groupby('customer')['revenue'].mean().sort_values(ascending=False))
# Q19
pd.pivot_table(sales_clean, values='revenue', index='product', columns='month', aggfunc='sum', fill_value=0)

In [ ]:
# Q20
left = pd.merge(sales_clean, customers, on='customer', how='left')
print(left[left['city'].isnull()][['customer', 'product']])
# Q21
inner = pd.merge(sales_clean, customers, on='customer', how='inner')
print("Inner rows:", len(inner))
# Q22
outer = pd.merge(sales_clean, customers, on='customer', how='outer', indicator=True)
print(outer[outer['_merge'] == 'right_only'][['customer', 'city']])

In [ ]:
# Q23
(sales.assign(qty=lambda d: d['qty'].fillna(d['qty'].median()))
      .assign(revenue=lambda d: d['qty'] * d['price'])
      .query("product in ['Phone', 'Laptop']")
      .groupby('product')['revenue'].sum())

In [ ]:
# Q24
print(sales_clean.groupby('customer')['revenue'].sum().idxmax())
# Q25
pd.read_csv('sample_data/emp.csv', index_col='id', usecols=['id', 'name', 'joined'], parse_dates=['joined'])

---
### 🎉 Next steps
- Official docs: https://pandas.pydata.org/docs/
- Practice on real datasets (Kaggle, `seaborn.load_dataset`)
- Learn Matplotlib/Seaborn for visualisation and scikit-learn for modelling

> **Cheat-sheet reminder:** Mean → symmetric numeric · Median → skewed/outlier numeric · Mode → categorical · `how='left'` keeps all left rows · `loc` = labels, `iloc` = positions.